# Modules

In [1]:
import pandas as pd
import html

# Verification initiale des datas

In [2]:
# On setup tout les CSV
CSVProduct = pd.read_csv('datasets/products.csv')
CSVDepartement = pd.read_csv('datasets/departments.csv')
CSVAisles = pd.read_csv('datasets/aisles.csv')
CSVOrderProduct = pd.read_csv('datasets/order_products.csv')
CSVOrder = pd.read_csv('datasets/orders.csv')


# Permet de voir le type des column, parfait pour voir les incoherences de type
for csv in [CSVAisles, CSVProduct, CSVDepartement, CSVOrderProduct, CSVOrder]:
    a = csv.info()
    for i in range(2):
        print("")


<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


<class 'pandas.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49688 non-null  int64
 1   product_name   49688 non-null  str  
 2   aisle_id       49688 non-null  int64
 3   department_id  49688 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 468.0 bytes


<c

# Verificatio des column clean

In [3]:
CSV_Clean = pd.read_csv('CSV_Cleaned.csv')
print(CSV_Clean)
# Permet de voir le type des column, parfait pour voir les incoherences de type
print(CSV_Clean.info())


         department                aisle                    product_name  \
0            frozen  frozen meat seafood              Ground Turkey Chub   
1            frozen  frozen meat seafood              Ground Turkey Chub   
2            frozen  frozen meat seafood              Ground Turkey Chub   
3            frozen  frozen meat seafood              Ground Turkey Chub   
4            frozen  frozen meat seafood              Ground Turkey Chub   
...             ...                  ...                             ...   
13690535    missing              missing  Lemon Cayenne Drinking Vinegar   
13690536    missing              missing  Lemon Cayenne Drinking Vinegar   
13690537    missing              missing  Lemon Cayenne Drinking Vinegar   
13690538    missing              missing  Lemon Cayenne Drinking Vinegar   
13690539    missing              missing                K Cup Dark Blend   

          order_id   user_id  order_number  order_dow  order_hour_of_day  \
0          

# Analyse coherence des valeurs

## Analyse coherence de type

In [39]:
# On va maintenant verifier que nous n'avons pas de float invalid, comme ce sont des id ils doivent absolument etre sous la forme X.0, si nous avons autre chose que .0 alors c'est un id invalid
# Pour cela nous allons compter le nombre de unique en float et le nombre de unique en integer, si c'est le meme nombre alors aucun conflit et ils sont tous different et corrrecte
# Sinon cela veut dire qu'il va y avoir un doublons donc un id sera present deux fois


#--------------------------------------------------------------------------------------------------------------------------#
# nunique --> permet de compter le nombre de valeur distinctes
# .between --> permet de comparer si un element est entre deux valeurs
# .any() --> permet d'arreter la boucle quand une valeur qui rempli la condition est trouve
# select_dtypes() --> permet de selectionner uniquement les valeurs qui respecte la condtion entre `()`
#--------------------------------------------------------------------------------------------------------------------------#


## On verifie pour user_id

In [48]:
Verif_before = CSV_Clean['user_id'].nunique()
print(f"nombre user_id float unique : {Verif_before}")
Verif_after = CSV_Clean['user_id'].astype(int).nunique()
print(f"nombre user_id integer unique : {Verif_after}")

nombre user_id float unique : 197361
nombre user_id integer unique : 197361


## On verifier pour order_id

In [49]:
Verif_before = CSV_Clean['order_id'].nunique()
print(f"nombre order_id float unique : {Verif_before}")
Verif_after = CSV_Clean['order_id'].astype(int).nunique()
print(f"nombre order_id integer unique : {Verif_after}")

nombre order_id float unique : 1357445
nombre order_id integer unique : 1357445


## On va donc verifier maintenant si oder_hour_od_day est bien entre 0 et 24 comme ce sont des heures

In [50]:
isValid = CSV_Clean['order_hour_of_day'].between(0, 24).all()
if isValid:
    print("Toute les valeurs sont comprises entre 0 et 24")
else:
    print("Probleme de coherence dans nos valeurs")

Toute les valeurs sont comprises entre 0 et 24


## Idem on va verifier si order_dow est bien entre 0 et 6

In [51]:
isValid = CSV_Clean['order_dow'].between(0, 24).all()
if isValid:
    print("Toute les valeurs sont comprises entre lundi et dimanche")
else:
    print("Probleme de coherence dans nos valeurs")

Toute les valeurs sont comprises entre lundi et dimanche


## Nous allons ensuite verifier si le dataset, contient des valeurs chiffres negatif

In [52]:
getNegatif = (CSV_Clean.select_dtypes(include='number').values < 0).any()
if getNegatif:
    print("Certaines valeurs sont negatives, probleme !")
else:
    print("Toute les valeurs sont <=0, pas de probleme")

Toute les valeurs sont <=0, pas de probleme


## On va maintenant verifier que department contient uniquement nos departement valide

In [53]:
departement_valid = CSVDepartement['department'].unique().tolist()
isValid = CSV_Clean['department'].isin(departement_valid).all()
if isValid:
    print("Tout les departement sont correcte")
else:
    print("Probleme des departement ne sont pas conforme")

Tout les departement sont correcte


## On va faire de meme pour les aisles

In [46]:
aisle_valid = CSVAisles['aisle'].unique().tolist()
isValid = CSV_Clean['aisle'].isin(aisle_valid).all()
if isValid:
    print("Tout les ailes sont correcte et valide")
else:
    print("Probleme d'ailes noms incorrecte")

Tout les ailes sont correcte et valide


## On va maintenant verifier la conformite des product_name

In [18]:
# Lors de nos manipulation nous avons deja pu observer que certains nom sont incorrecte et necessite des transformation

# nous voyons tout d'abord qu'il y a des code html present dans les nom
CSV_Clean['product_name'] = CSV_Clean['product_name'].apply(html.unescape)
isEncore = CSV_Clean[CSV_Clean['product_name'].str.contains('&#')]

if isEncore.empty:
    print("Tout les `&#` ont ete change avec succes")
else:
    print(f"Pas ok, il reste {len(isEncore)} a changer")


# Affiche uniquement la colonne 'product_name' pour ces lignes
# print(CSV_Clean.loc[CSV_Clean['product_name'].str.contains('#'), 'product_name']

Tout les `&#` ont ete change avec succes
948952                            The World's Best Pencil #2
948953                            The World's Best Pencil #2
948954                            The World's Best Pencil #2
948955                            The World's Best Pencil #2
948956                            The World's Best Pencil #2
                                 ...                        
5649746    Super Premium White Paper #4 Size Cone Coffee ...
5649747    Super Premium White Paper #4 Size Cone Coffee ...
5689865                                    #2 Coffee Filters
5689866                                    #2 Coffee Filters
5689867                                    #2 Coffee Filters
Name: product_name, Length: 80, dtype: str
